In [1]:
import sys
import os

# Detect environment
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

# Environment-specific settings
if IS_COLAB:
    print("Running on Google Colab")
    ENV_NAME = "colab"
    USE_GPU = True
    TESSERACT_PATH = None  
    
elif IS_KAGGLE:
    print("Running on Kaggle")
    ENV_NAME = "kaggle"
    USE_GPU = True
    TESSERACT_PATH = None  
    
else:  # Local
    print("Running Locally")
    ENV_NAME = "local"
    USE_GPU = False  
    TESSERACT_PATH = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

print(f"Environment: {ENV_NAME}")
print(f"GPU enabled: {USE_GPU}")

Running Locally
Environment: local
GPU enabled: False


# Environment Configuration

Auto-detect running environment (Local, Kaggle, or Google Colab)

# Installation

In [ ]:
# Fix NumPy version compatibility
!pip install -q "numpy<2.0"

!pip install -q pymupdf layoutparser torch torchvision
!pip install -q "detectron2@git+https://github.com/facebookresearch/detectron2.git@v0.6"
!pip install -q "git+https://github.com/Layout-Parser/layout-parser.git"

# Fix Pillow version for compatibility with torchvision
!pip install -q "Pillow>=10.0.0"

# OCR engines - only VietOCR
!pip install -q vietocr

# PhoBERT dependencies
!pip install -q transformers
!pip install -q scikit-learn


^C



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Libraries

In [2]:
import os
import fitz  # PyMuPDF
import cv2
import re
import requests
import json
from urllib.parse import urlparse
from datetime import datetime
import numpy as np
from PIL import Image
import pytesseract

# OCR engines - VietOCR only
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

import torch
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity


# Initialize Models

In [3]:
if 'USE_GPU' not in globals():
    USE_GPU = False
    TESSERACT_PATH = None

# Set Tesseract path for Windows
if TESSERACT_PATH:
    import pytesseract
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH
    print(f"Tesseract path set: {TESSERACT_PATH}")
else:
    # Try default Windows path
    import pytesseract
    pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Initialize VietOCR (for Vietnamese text recognition)
print("Loading VietOCR...")
try:
    if 'viet_ocr' not in globals():
        config = Cfg.load_config_from_name('vgg_transformer')
        import torch
        device_vietocr = 'cuda' if (USE_GPU and torch.cuda.is_available()) else 'cpu'
        config['device'] = device_vietocr
        config['predictor']['beamsearch'] = False
        viet_ocr = Predictor(config)
        print(f"VietOCR loaded on {device_vietocr}")
    else:
        print(f"VietOCR already loaded (reusing existing instance)")
except Exception as e:
    print(f"VietOCR error: {type(e).__name__}: {str(e)}")
    raise

# Initialize PhoBERT
print("Loading PhoBERT...")
try:
    if 'phobert' not in globals():
        import torch
        device = torch.device('cuda' if (USE_GPU and torch.cuda.is_available()) else 'cpu')
        phobert = AutoModel.from_pretrained("vinai/phobert-base")
        tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
        phobert.eval()
        phobert.to(device)
        print(f"PhoBERT loaded on {device}")
    else:
        print(f"PhoBERT already loaded (reusing existing instance)")
except Exception as e:
    print(f"PhoBERT error: {type(e).__name__}: {str(e)}")
    raise

Tesseract path set: C:\Program Files\Tesseract-OCR\tesseract.exe
Loading VietOCR...


c:\Users\Xuan Nhi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Model weight C:\Users\XUANNH~1\AppData\Local\Temp\vgg_transformer.pth exsits. Ignore download!
VietOCR loaded on cpu
Loading PhoBERT...
PhoBERT loaded on cpu


# PhoBERT Processing Functions

In [4]:
def get_text_embedding(text, max_length=256):
    """
    Encode Vietnamese text using PhoBERT
    Returns sentence embedding (average of token embeddings)
    """
    if not text or len(text.strip()) == 0:
        return np.zeros((1, 768))  # PhoBERT embedding size
    
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors='pt',
        max_length=max_length,
        truncation=True,
        padding='max_length'
    ).to(device)
    
    # Get embeddings
    with torch.no_grad():
        outputs = phobert(**inputs)
        # Use mean pooling over token embeddings
        embeddings = outputs.last_hidden_state.mean(dim=1)
    
    return embeddings.cpu().numpy()


def calculate_similarity(text1, text2):
    """
    Calculate cosine similarity between two Vietnamese texts
    """
    emb1 = get_text_embedding(text1)
    emb2 = get_text_embedding(text2)
    
    similarity = cosine_similarity(emb1, emb2)[0][0]
    return float(similarity)

In [ ]:
class phobertProcessing():
    def classify_problem_type(question_text):
        """
        Classify problem type using HYBRID approach:
        1. Rule-based detection for strong geometry signals (PRIORITY)
        2. PhoBERT semantic similarity (FALLBACK)
        """
        question_lower = question_text.lower()
        
        # === 1. STRONG GEOMETRY SIGNALS (Rule-based override) ===
        # These terms are DEFINITIVE geometry indicators
        strong_geometry_terms = [
            'tam giác',       # triangle (very strong)
            'tứ giác',        # quadrilateral
            'đường tròn',     # circle
            'định lí thalès', 'định lý thalès', 'thalès',  # Thales theorem
            'định lí pythagore', 'định lý pythagore', 'pythagore',  # Pythagoras theorem
            'góc đối đỉnh',   # vertical angles
            'góc nội tiếp',   # inscribed angle
            'góc ở tâm',      # central angle
            'song song',      # parallel
            'vuông góc',      # perpendicular
            'đồng dạng',      # similar (geometry)
            'đồng qui',       # concurrent
            'trung tuyến',    # median
            'phân giác',      # angle bisector
            'đường cao',      # altitude
            'chu vi',         # perimeter
            'diện tích tam giác', 'diện tích tứ giác',  # area (with shape)
            'hình bình hành', # parallelogram
            'hình thang',     # trapezoid
            'hình chữ nhật',  # rectangle
            'hình vuông',     # square
            'hình thoi',      # rhombus
            'cạnh huyền',     # hypotenuse
            'cạnh góc vuông', # leg of right triangle
            'đường chéo',     # diagonal
            'bán kính',       # radius
            'đường kính',     # diameter
            'tiếp tuyến',     # tangent
            'cát tuyến',      # secant
            'dây cung',       # chord
            'góc nhọn', 'góc tù', 'góc vuông',  # angle types
            'đỉnh', 'cạnh', 'đáy',  # vertices, sides, base
        ]
        
        # If ANY strong geometry term found → FORCE geometry classification
        for term in strong_geometry_terms:
            if term in question_lower:
                return 'geometry', {'geometry': 1.0}
        
        # === 2. PHOBERT SEMANTIC SIMILARITY (Fallback) ===
        # Enhanced keywords with more geometry coverage
        problem_types = {
            'geometry': '''
                hình học tam giác tứ giác đường tròn góc đoạn thẳng 
                chu vi diện tích định lí Thalès định lí Pythagore
                song song vuông góc đồng dạng trung tuyến phân giác
                đường cao cạnh góc đỉnh hình bình hành hình thang
                hình chữ nhật hình vuông hình thoi cạnh huyền
            ''',
            'algebra': 'phương trình bất phương trình hệ phương trình biến số nghiệm ẩn số giải',
            'number_theory': 'số nguyên chia hết ước số bội số số chính phương số thập phân phân số',
            'measurement': 'đo lường đơn vị chuyển đổi tính toán độ dài thời gian khối lượng',
            'word_problem': 'bài toán thực tế ứng dụng vận tốc quãng đường tiền lương giá cả'
        }
        
        # Get embedding for question
        question_emb = get_text_embedding(question_text)
        
        # Compare with each type
        scores = {}
        for ptype, keywords in problem_types.items():
            type_emb = get_text_embedding(keywords)
            similarity = cosine_similarity(question_emb, type_emb)[0][0]
            scores[ptype] = float(similarity)
        
        # Get best match
        best_type = max(scores, key=scores.get)
        
        return best_type, scores
    
    
    def analyze_figure_relevance(question_text, figure_refs):
        """
        PhoBERT-first approach: Semantic analysis is primary signal
        Rule-based keywords are secondary validation
        """
        # === 1. SEMANTIC ANALYSIS (PRIMARY - 70%) ===
        # Expanded context for better semantic matching
        figure_context = (
            "hình vẽ minh họa đồ thị biểu đồ sơ đồ bản vẽ "
            "hình ảnh quan sát theo hình trong hình dựa vào hình"
        )
        semantic_score = calculate_similarity(question_text, figure_context)
        
        # === 2. EXPLICIT KEYWORDS (SECONDARY - 20%) ===
        figure_keywords = [
            'trong hình', 'hình vẽ', 'theo hình', 'quan sát hình',
            'từ hình', 'dựa vào hình', 'xem hình', 'cho hình'
        ]
        explicit_mention = any(keyword in question_text.lower() 
                              for keyword in figure_keywords)
        
        # === 3. FIGURE REFERENCES (TERTIARY - 10%) ===
        has_figure_refs = len(figure_refs) > 0
        
        # === 4. WEIGHTED COMBINATION ===

        # PhoBERT semantic now dominates (70% vs old 30%)        return min(relevance, 1.0)

        relevance = (        

            0.70 * semantic_score +                       # PhoBERT (PRIMARY)        )

            0.20 * (1.0 if explicit_mention else 0) +     # Keywords (SECONDARY)            0.10 * (1.0 if has_figure_refs else 0)        # References (TERTIARY)

# Utils - PDF Processing

In [27]:
def download_pdf_from_url(url, save_dir="input", chunk_size=8192):
    os.makedirs(save_dir, exist_ok=True)
    filename = os.path.basename(urlparse(url).path)
    if not filename.endswith(".pdf"):
        filename = "document.pdf"

    save_path = os.path.join(save_dir, filename)
    if os.path.exists(save_path):
        print(f"PDF already exists: {save_path}")
        return save_path

    print(f"Download PDF...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in r.iter_content(chunk_size):
                if chunk:
                    f.write(chunk)
    
    print(f"Saved: {save_path}")
    return save_path


def pdf_to_images(pdf_path, out_dir="pdf_pages", dpi=200):
    os.makedirs(out_dir, exist_ok=True)
    doc = fitz.open(pdf_path)

    image_paths = []
    for i, page in enumerate(doc):
        pix = page.get_pixmap(dpi=dpi)
        img_path = f"{out_dir}/page_{i+1:03d}.png"
        pix.save(img_path)
        image_paths.append(img_path)

    print(f"Converted {len(image_paths)} pages to images")
    return image_paths

# OCR Processing

In [28]:
def preprocess_image_for_ocr(image_path):
    # Read image
    img = cv2.imread(image_path)
    
    # 1. Upscale if needed (for low-res scans)
    height, width = img.shape[:2]
    if width < 2000:  # If width < 2000px, upscale
        scale = 2000 / width
        img = cv2.resize(img, None, fx=scale, fy=scale, 
                        interpolation=cv2.INTER_CUBIC)
    
    # 2. Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 3. Denoise (remove scanner noise and artifacts)
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
    
    # 4. Increase contrast using CLAHE (Contrast Limited Adaptive Histogram Equalization)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    contrast = clahe.apply(denoised)
    
    # 5. Binarization - Otsu's method works well for printed text
    _, binary = cv2.threshold(contrast, 0, 255, 
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # 6. Morphological opening (remove small noise dots)
    kernel = np.ones((2, 2), np.uint8)
    opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
    
    return opened

In [29]:
def clean_math_symbols(text):
    """
    Advanced post-processing with pattern-based corrections
    Specifically tuned for Vietnamese math textbooks
    """
    
    # === 0. PARALLEL SYMBOL FIXES ===
    parallel_mistakes = [
        (r'⁄⁄', '//'),  
        (r'⁄\s*⁄', '//'),
        (r'/\s*/', '//'),
        (r'ỹ', '//'),   
        (r'ỉ', '//'),
        (r'ý', '//'),
    ]
    
    for pattern, replacement in parallel_mistakes:
        text = re.sub(pattern, replacement, text)

    # === 1. DEGREE SYMBOL CORRECTIONS ===
    # Context-aware: number + ? or 0 → °
    text = re.sub(r'(\d+)\s*[?O]\s*(?=[A-Z]|nên|;|,|\)|$)', r'\1°', text)
    text = re.sub(r'(180|90|45|30|60|120|360)\s*0\s*(?=[A-Z]|nên|;|,|\)|$)', r'\1°', text)
    
    # === 2. ANGLE FORMULAS ===
    angle_formulas = [
        # (180? - A) → (180° - Â)
        (r'\((\d+)\s*[?0]\s*-\s*([A-Z])\)', r'(\1° - \2̂)'),
        # 4.180? → 4 · 180°
        (r'(\d+)\s*\.\s*(\d+)\s*[?0]', r'\1 · \2°'),
        # 2.360 - 360 → 2 · 360° - 360°
        (r'2\s*\.\s*360\s*[?0]?\s*-\s*360\s*[?0]?', r'2 · 360° - 360°'),
        # 4.180 - (A+B+C+D) → 4 · 180° - (Â + B̂ + Ĉ + D̂)
        (r'4\s*\.\s*180\s*[?0]?\s*-\s*\(([A-Z])\s*\+\s*([A-Z])\s*\+\s*([A-Z])\s*\+\s*([A-Z])\)', 
         r'4 · 180° - (\1̂ + \2̂ + \3̂ + \4̂)'),
    ]
    
    for pattern, replacement in angle_formulas:
        text = re.sub(pattern, replacement, text)
    
    # === 3. GEOMETRY TERMS ===
    # Shape names with symbols
    geometry_terms = {
        r'tam giác\s+([A-Z]{3})': r'△\1',
        r'tứ giác\s+([A-Z]{4})': r'▱\1',
        r'góc\s+([A-Z]{1,3})': r'∠\1',
    }
    
    for pattern, replacement in geometry_terms.items():
        text = re.sub(pattern, replacement, text)
    
    # === 4. ANGLE VERTEX CORRECTIONS ===
    # Add circumflex to capital letters in math expressions
    # Pattern: isolated capital letter followed by math operators
    text = re.sub(r'\b([A-Z])\b(?=\s*[+\-=<>]|\s*\)|\s*,|\s*nên)', r'\1̂', text)
    
    # === 5. MATH OPERATORS ===
    operator_corrections = {
        ' - ': ' − ',      
        '>=': '≥',
        '<=': '≤',
        '!=': '≠',
        '~=': '≈',
        'x ': '× ',        
        ' x ': ' × ',
    }
    
    for old, new in operator_corrections.items():
        text = text.replace(old, new)
    
    # === 6. VIETNAMESE MATH TERMS TO SYMBOLS ===
    vietnamese_terms = {
        'xấp xỉ': '≈',
        'lớn hơn hoặc bằng': '≥',
        'nhỏ hơn hoặc bằng': '≤',
        'khác': '≠',
    }
    
    for vn_term, symbol in vietnamese_terms.items():
        text = text.replace(vn_term, symbol)
    
    return text

In [30]:
def ocr_page_tesseract(image_path):
    """
    OCR using Tesseract 
    VietOCR hybrid fails with math formulas, Tesseract alone is more reliable
    """
    try:
        img = Image.open(image_path)
        
        # Best config for Vietnamese math textbooks
        config = '--oem 3 --psm 3 -l vie'
        text = pytesseract.image_to_string(img, config=config)
        
        # Clean up whitespace
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\n\s+\n', '\n\n', text)
        
        # Apply math symbol corrections
        text = clean_math_symbols(text)
        
        return text
    except Exception as e:
        print(f"Tesseract error: {e}")
        return ""

In [31]:
def normalize_for_training(text):
    """
    Convert special math symbols back to plain Vietnamese text
    This version is suitable for training ML models
    """
    # === 1. SYMBOLS TO VIETNAMESE ===
    symbol_to_vietnamese = {
        # Degree symbol
        '°': ' độ',
        
        # Geometry symbols
        '∠': 'góc ',
        '△': 'tam giác ',
        '▱': 'tứ giác ',
        '□': 'hình vuông ',
        '○': 'đường tròn ',
        
        # Math operators
        # '×': ' nhân ',
        '÷': ' chia ',
        '±': ' cộng trừ ',
        '≈': ' xấp xỉ ',
        '≠': ' khác ',
        '≥': ' lớn hơn hoặc bằng ',
        '≤': ' nhỏ hơn hoặc bằng ',
        '≡': ' đồng nhất ',
        '∞': ' vô cùng ',
        
        # Special operators
        '·': ' nhân ',  # dot multiplication
        '−': ' trừ ',   # minus sign (not hyphen)
        '√': 'căn bậc hai ',
        '∑': 'tổng ',
        '∫': 'tích phân ',
    }
    
    for symbol, vietnamese in symbol_to_vietnamese.items():
        text = text.replace(symbol, vietnamese)
    
    # === 2. REMOVE CIRCUMFLEX FROM ANGLE VERTICES ===
    # Â, B̂, Ĉ, D̂, etc. → A, B, C, D
    circumflex_chars = {
        'Â': 'A', 'B̂': 'B', 'Ĉ': 'C', 'D̂': 'D', 'Ê': 'E',
        'F̂': 'F', 'Ĝ': 'G', 'Ĥ': 'H', 'Î': 'I', 'Ĵ': 'J',
        'K̂': 'K', 'L̂': 'L', 'M̂': 'M', 'N̂': 'N', 'Ô': 'O',
        'P̂': 'P', 'Q̂': 'Q', 'R̂': 'R', 'Ŝ': 'S', 'T̂': 'T',
        'Û': 'U', 'V̂': 'V', 'Ŵ': 'W', 'X̂': 'X', 'Ŷ': 'Y', 'Ẑ': 'Z',
    }
    
    for circumflex, plain in circumflex_chars.items():
        text = text.replace(circumflex, plain)
    
    # === 3. CLEAN UP EXTRA SPACES ===
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text

In [32]:
def ocr_page_hybrid_tesseract_vietocr(image_path, use_preprocessing=True):
    """
    HYBRID APPROACH: Tesseract detects text regions, VietOCR recognizes Vietnamese
    - Tesseract: Good at detecting text layout and bounding boxes
    - VietOCR: Better at recognizing Vietnamese characters correctly
    - Optional preprocessing for better accuracy (especially for math formulas)
    """
    try:
        if use_preprocessing:
            preprocessed = preprocess_image_for_ocr(image_path)
            img = Image.fromarray(preprocessed)
            img_cv = preprocessed
            # Convert back to RGB for VietOCR
            img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_GRAY2RGB)
        else:
            # Load image without preprocessing
            img = Image.open(image_path)
            img_cv = cv2.imread(image_path)
            img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
        
        # Step 1: Use Tesseract to detect text regions (get bounding boxes)
        # Output_type=Output.DICT returns dict with bbox coordinates
        ocr_data = pytesseract.image_to_data(img, lang='vie', output_type=pytesseract.Output.DICT)
        
        # Step 2: Extract valid text boxes
        text_blocks = []
        n_boxes = len(ocr_data['text'])
        
        for i in range(n_boxes):
            # Filter out empty or low-confidence detections
            conf = int(ocr_data['conf'][i]) if ocr_data['conf'][i] != '-1' else 0
            text = ocr_data['text'][i].strip()
            
            # Skip if confidence too low or empty
            if conf < 30 or len(text) == 0:
                continue
            
            # Get bounding box from Tesseract
            x = ocr_data['left'][i]
            y = ocr_data['top'][i]
            w = ocr_data['width'][i]
            h = ocr_data['height'][i]
            
            # Skip very small boxes
            if w < 20 or h < 10:
                continue
            
            # Add padding
            padding = 5
            x = max(0, x - padding)
            y = max(0, y - padding)
            w = min(img_cv.shape[1] - x, w + 2*padding)
            h = min(img_cv.shape[0] - y, h + 2*padding)
            
            # Crop region
            cropped = img_rgb[y:y+h, x:x+w]
            
            if cropped.size == 0:
                continue
            
            # Step 3: Use VietOCR to recognize Vietnamese text in cropped region
            try:
                cropped_pil = Image.fromarray(cropped)
                vietocr_text = viet_ocr.predict(cropped_pil)
                
                if vietocr_text and len(vietocr_text.strip()) > 0:
                    text_blocks.append({
                        'text': vietocr_text,
                        'y': y,
                        'x': x,
                        'line_num': ocr_data['line_num'][i],
                        'block_num': ocr_data['block_num'][i]
                    })
            except Exception as e:
                # Fallback to Tesseract text if VietOCR fails
                text_blocks.append({
                    'text': text,
                    'y': y,
                    'x': x,
                    'line_num': ocr_data['line_num'][i],
                    'block_num': ocr_data['block_num'][i]
                })
        
        # Step 4: Sort by block, line, then x position (reading order)
        text_blocks.sort(key=lambda b: (b['block_num'], b['line_num'], b['x']))
        
        # Step 5: Merge text with proper spacing and line breaks
        final_text = []
        prev_block = None
        prev_line = None
        
        for block in text_blocks:
            # Add paragraph break between blocks
            if prev_block is not None and block['block_num'] != prev_block:
                final_text.append('\n\n')
            # Add line break between lines in same block
            elif prev_line is not None and block['line_num'] != prev_line:
                final_text.append('\n')
            # Add space between words on same line
            elif len(final_text) > 0 and final_text[-1] not in ['\n', '\n\n']:
                final_text.append(' ')
            
            final_text.append(block['text'])
            prev_block = block['block_num']
            prev_line = block['line_num']
        
        full_text = ''.join(final_text)
        
        # Clean up whitespace
        full_text = re.sub(r'[ \t]+', ' ', full_text)
        full_text = re.sub(r'\n\s+\n', '\n\n', full_text)
        
        # Fix math symbols
        full_text = clean_math_symbols(full_text)
        
        return full_text
        
    except Exception as e:
        print(f"Hybrid OCR error: {e}")
        return ""

# Problem Extraction with PhoBERT

In [41]:
def extract_problems_with_figures_phobert(text, min_figure_relevance=0.0):
    """
    Extract problems using PhoBERT for better Vietnamese understanding
    Enhanced version with semantic analysis
    
    Args:
        text: OCR text to extract problems from
        min_figure_relevance: Minimum figure relevance score (0.0-1.0) to include problem.
                             Set to 0.0 to extract all problems, then filter later.
    """
    figure_patterns = [
        # Numbered figures with various formats
        (r'Hình\s*\d+\.\d+', 'Hình X.Y'),
        (r'\(H\.\d+\.\d+\)', '(H.X.Y)'),
        (r'H\.\d+\.\d+', 'H.X.Y'),
        (r'hình\s*\d+\.\d+', 'hình X.Y'),
        
        # Reference to figures in text
        (r'trong\s+(?:các\s+)?hình\s+(?:vẽ\s+)?(?:sau|trên|dưới|bên|\d)', 'trong hình...'),
        (r'theo\s+hình\s+(?:vẽ\s+)?(?:sau|trên|dưới|bên|\d)', 'theo hình...'),
        (r'xem\s+hình\s+(?:vẽ\s+)?(?:sau|trên|dưới|bên|\d)', 'xem hình...'),
        (r'hình\s+(?:vẽ\s+)?(?:sau|dưới|bên|minh\s+họa)', 'hình...'),
        
        # Common phrases
        (r'cho\s+(?:các\s+)?hình', 'cho hình'),
        (r'có\s+hình\s+vẽ', 'có hình vẽ'),
    ]
    
    # Keywords to filter out chart/graph problems
    chart_keywords = [
        r'biểu\s*đồ',
        r'đồ\s*thị',
        r'chart',
        r'graph',
        r'bảng\s*thống\s*kê',
    ]
    
    results = []
    all_markers = []
    
    # Find numbered problems
    for match in re.finditer(r'(?:^|\n|(?<=\s{2}))(\d+\.\d+)\.?\s*', text, re.MULTILINE):
        problem_num = match.group(1)
        
        if match.start() > 0:
            before = text[max(0, match.start()-5):match.start()]
            if re.search(r'[a-zđáàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵ]\s*$', before, re.IGNORECASE):
                continue
        
        all_markers.append({
            'type': 'numbered',
            'identifier': problem_num,
            'start': match.start(),
            'end': match.end(),
            'full_match': match.group(0).strip()
        })
    
    # Find example problems
    for match in re.finditer(r'Ví\s+dụ\s+(\d+)', text, re.IGNORECASE):
        all_markers.append({
            'type': 'example',
            'identifier': f'VD{match.group(1)}',
            'start': match.start(),
            'end': match.end(),
            'full_match': match.group(0)
        })
    
    # Sort and deduplicate markers
    all_markers.sort(key=lambda x: x['start'])
    unique_markers = []
    last_pos = -100
    for marker in all_markers:
        if marker['start'] - last_pos > 5:
            unique_markers.append(marker)
            last_pos = marker['start']
    
    all_markers = unique_markers
    
    if not all_markers:
        return results
    
    # Track figures seen on this page to avoid duplicates
    seen_figures = {}  # {figure_name: first_problem_num}
    
    # Process each problem segment
    for i, marker in enumerate(all_markers):
        start_pos = marker['end']
        end_pos = all_markers[i+1]['start'] if i+1 < len(all_markers) else len(text)
        
        segment = text[start_pos:end_pos].strip()
        
        if len(segment) < 20:
            continue
        
        # Filter out chart/graph problems
        is_chart_problem = any(re.search(pattern, segment, re.IGNORECASE) for pattern in chart_keywords)
        if is_chart_problem:
            continue
        
        # Find figures
        found_figures = []
        figure_positions = []
        
        for fig_pattern, _ in figure_patterns:
            for match in re.finditer(fig_pattern, segment, re.IGNORECASE):
                fig_name = match.group(0)
                found_figures.append(fig_name)
                figure_positions.append(match.start())
        
        # Remove duplicate figures already seen in previous problems
        unique_figures = []
        for fig in found_figures:
            if fig not in seen_figures:
                seen_figures[fig] = marker['identifier']
                unique_figures.append(fig)
        
        found_figures = unique_figures if unique_figures else found_figures
        
        # Handle merged problems
        if i+1 < len(all_markers):
            next_marker_pattern = all_markers[i+1]['identifier']
            next_match = re.search(rf'\b{re.escape(next_marker_pattern)}\b', segment)
            
            if next_match:
                next_pos = next_match.start()
                valid_figures = [fig for fig, pos in zip(found_figures, figure_positions) if pos < next_pos]
                
                if not valid_figures:
                    continue
                
                found_figures = valid_figures
                segment = segment[:next_pos].strip()
        
        # Split question and solution
        giai_match = re.search(r'\bGiải\b', segment, re.IGNORECASE)
        
        if giai_match:
            question_part = segment[:giai_match.start()].strip()
            solution_part = segment[giai_match.end():].strip()
            has_solution = True
        else:
            question_part = segment
            solution_part = ""
            has_solution = False
        
        # Clean text
        question_part = re.sub(r'^[-@•()\s=:;]+', '', question_part).strip()
        solution_part = re.sub(r'^[-@•()\s=:;]+', '', solution_part).strip()
        
        if len(question_part) < 30:
            continue
        
        question_part = re.sub(r'\s+', ' ', question_part)
        
        # === PhoBERT Analysis ===
        print(f"Analyzing {marker['identifier']} with PhoBERT...", end=" ")
        
        # Classify problem type
        prob_type, type_scores = phobertProcessing.classify_problem_type(question_part)
        
        # Analyze figure relevance
        fig_relevance = phobertProcessing.analyze_figure_relevance(question_part, found_figures)
        
        # Get embedding for similarity search later
        embedding = get_text_embedding(question_part)
        
        print(f"Type: {prob_type}, Fig relevance: {fig_relevance:.2f}")
        
        # Filter by minimum figure relevance threshold
        if fig_relevance < min_figure_relevance:
            continue
        
        results.append({
            'problem_number': marker['identifier'],
            'problem_type': marker['type'],
            'content': segment,
            'question': question_part,
            'solution': solution_part,
            'figures': list(set(found_figures)),
            'has_solution': has_solution,
            'embedding': embedding.tolist()[0],  # Store for similarity search
            # PhoBERT features
            'figure_relevance': fig_relevance,
            'ai_problem_type': prob_type,
            'type_confidence': type_scores,
        })
    
    return results

# Download and Process PDF

In [16]:
PDF_URL = "https://84864e12bc.vws.vegacdn.vn//data/doc/2025/thcslienninh/2025_2/26/sach-bai-tap-toan-8-tap-1-ket-noi-tri-thuc-voi-cuoc-song_262202515.pdf"

pdf_file_path = download_pdf_from_url(PDF_URL, save_dir="./input")
page_images = pdf_to_images(pdf_file_path, out_dir="output/pages", dpi=400) 

PDF already exists: ./input\sach-bai-tap-toan-8-tap-1-ket-noi-tri-thuc-voi-cuoc-song_262202515.pdf
Converted 113 pages to images


# Process Pages with OCR

In [34]:
import gc

pages_dir = os.path.join("output", "pages")

if not os.path.exists(pages_dir):
    print(f"Not found: {pages_dir}")
else:
    # Get all page image files
    page_images = sorted([
        os.path.join(pages_dir, f) 
        for f in os.listdir(pages_dir) 
        if f.endswith('.png')
    ])
    
    print(f"Processing {len(page_images)} pages with TESSERACT OCR")
    print("--------Using Tesseract--------\n")
    
    # OCR each page
    page_texts = {}
    
    for idx, page_img in enumerate(page_images):
        page_num = idx + 1
        print(f"[{page_num}/{len(page_images)}] {os.path.basename(page_img)}...", end=" ", flush=True)
        
        try:
            # Use Tesseract 
            text = ocr_page_tesseract(page_img)
            page_texts[page_num] = text
            print(f"({len(text)} chars)")
            
            # Clear memory every 10 pages
            if page_num % 10 == 0:
                gc.collect()
                
        except Exception as e:
            print(f"Error: {str(e)[:100]}")
            page_texts[page_num] = ""
            gc.collect()
            continue
    
    print(f"\nCompleted OCR for {len(page_texts)} pages")
    print(f"Pages with content: {sum(1 for t in page_texts.values() if len(t) > 0)}")
    print(f"Average chars/page: {sum(len(t) for t in page_texts.values()) / len(page_texts):.0f}")


Processing 113 pages with TESSERACT OCR
--------Using Tesseract--------

[1/113] page_001.png... (52 chars)
[2/113] page_002.png... (0 chars)
[3/113] page_003.png... (1682 chars)
[4/113] page_004.png... (692 chars)
[5/113] page_005.png... (1129 chars)
[6/113] page_006.png... (1214 chars)
[7/113] page_007.png... (1234 chars)
[8/113] page_008.png... (1021 chars)
[9/113] page_009.png... (1121 chars)
[10/113] page_010.png... (919 chars)
[11/113] page_011.png... (454 chars)
[12/113] page_012.png... (930 chars)
[13/113] page_013.png... (1083 chars)
[14/113] page_014.png... (522 chars)
[15/113] page_015.png... (1158 chars)
[16/113] page_016.png... (1242 chars)
[17/113] page_017.png... (1283 chars)
[18/113] page_018.png... (1392 chars)
[19/113] page_019.png... (751 chars)
[20/113] page_020.png... (907 chars)
[21/113] page_021.png... (686 chars)
[22/113] page_022.png... (882 chars)
[23/113] page_023.png... (561 chars)
[24/113] page_024.png... (596 chars)
[25/113] page_025.png... (737 chars)
[26

In [49]:
# Show OCR result for a specific page
page_to_show = 91

if page_to_show in page_texts:
    text = page_texts[page_to_show]
    print(f"PAGE {page_to_show} - OCR RESULT ({len(text)} characters)")
    print(text)
    print(f"END OF PAGE {page_to_show}")
else:
    print(f"Page {page_to_show} not found in results")
    print(f"Available pages: {sorted(page_texts.keys())}")

PAGE 91 - OCR RESULT (1164 characters)
313.

c) Do 142MVCB là hình bình hành nên WO / MB, từ đó NCB = MBA (hai góc
đồng vị). Điều kiện để hình thang /IMINCA là hình thang cân là ∠MAB = NCB
tức là MAB = MBA.

Vậy điều kiện đề J/IVCA là hình thang cân là tam giác AB cân tại /.

d)(H.3 · 1°b). Do 12VDO là hình bình hành

nên NÐ # MC, từ đó NDC = MCA. (hai M N
góc đồng vị). Điều kiện để hình thang
IMNDA là hình thang cân là NDC = MAB.
Vậy điều kiện đề AZNVDA là hình thang
cân là MMCA = MAB tức là △IMAC
cân tại I. Do B là đường trung tuyến của
tam giác 1AC nên điều kiện đề △MAC cân tại M là MB vuông góc với AC.
Vậy điều kiện đề hình thang WINVDA là hình thang cân đó là tam giác AB
vuông tại B.

(H.3.11). Xét hình thang ABCD với A à

hai đáy AB và CD. Giả sử AB < CD.

A B l@i D
Hình 3 · 1°b

Kẻ đường thẳắng đi qua 8 song song
với AD, nó cắt CD tại E thì ABED
là hình bình hành nên AB = DE; do ụ − g
AB < CDnên E nằm giữa C và D̂, do Hình 3.11

đó EC = DC — AB.

Ta có AD = BE nên trong △BEC,
BE

# Extract Problems with PhoBERT Analysis

In [45]:
print("Extracting problems with PhoBERT analysis...")

all_problems = []
for page_num, text in page_texts.items():
    problems = extract_problems_with_figures_phobert(text, min_figure_relevance=0.0)
    
    for prob in problems:
        prob['page_number'] = page_num
        all_problems.append(prob)

print(f"  Total problems extracted: {len(all_problems)}")
print(f"  Processed: {len(page_texts)} pages")

# Count by type
type_counts = {}
for p in all_problems:
    ptype = p.get('ai_problem_type', 'unknown')
    type_counts[ptype] = type_counts.get(ptype, 0) + 1

print(f"\n  Problem types breakdown:")
for ptype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"    {ptype}: {count}")

# Count geometry with/without figures
geometry_all = [p for p in all_problems if p.get('ai_problem_type') == 'geometry']
geometry_with_figures = [p for p in geometry_all if len(p.get('figures', [])) > 0]
geometry_no_figures = [p for p in geometry_all if len(p.get('figures', [])) == 0]

print(f"\n  Geometry problems:")
print(f"    Total geometry: {len(geometry_all)}")
print(f"    └─ With figures: {len(geometry_with_figures)}")
print(f"    └─ Without figures: {len(geometry_no_figures)}")

problems_with_figures = geometry_with_figures

print(f"\nFinal count: {len(problems_with_figures)} geometry problems with figures")

Extracting problems with PhoBERT analysis...
Analyzing 1.2 with PhoBERT... Type: geometry, Fig relevance: 0.23
Analyzing 1.3 with PhoBERT... Type: measurement, Fig relevance: 0.43
Analyzing 1.4 with PhoBERT... Type: geometry, Fig relevance: 0.22
Analyzing 1.5 with PhoBERT... Type: measurement, Fig relevance: 0.47
Analyzing 1.6 with PhoBERT... Type: number_theory, Fig relevance: 0.21
Analyzing 1.12 with PhoBERT... Type: geometry, Fig relevance: 0.25
Analyzing 1.14 with PhoBERT... Type: measurement, Fig relevance: 0.48
Analyzing 1.15 with PhoBERT... Type: measurement, Fig relevance: 0.48
Analyzing 1.16 with PhoBERT... Type: measurement, Fig relevance: 0.48
Analyzing 1.17 with PhoBERT... Type: number_theory, Fig relevance: 0.18
Analyzing 1.19 with PhoBERT... Type: number_theory, Fig relevance: 0.20
Analyzing 1.23 with PhoBERT... Type: geometry, Fig relevance: 0.21
Analyzing 1.25 with PhoBERT... Type: measurement, Fig relevance: 0.47
Analyzing 1.26 with PhoBERT... Type: word_problem, Fig r

In [46]:
if len(problems_with_figures) > 0:
    print(f"\nPREVIEW problems with figures")
    
    for i, prob in enumerate(problems_with_figures[:10], 1):
        print(f"[{i}] Problem {prob['problem_number']} (Page: {prob['page_number']})")
        print(f"    Type: {prob['ai_problem_type']}")
        print(f"    Figure relevance: {prob['figure_relevance']:.2f}")
        print(f"    Figures: {', '.join(prob['figures']) if prob['figures'] else 'None'}")
        
        print(f"    Question (normalized): {normalize_for_training(prob['question'])[:300]}...")
        print()


PREVIEW problems with figures
[1] Problem 2.24 (Page: 30)
    Type: geometry
    Figure relevance: 0.37
    Figures: H.2.4, Hình 2.4, (H.2.4)
    Question (normalized): Từ một miếng b//a có dạng hình tròn (H.2.4) với bán kính R(cm), người ta khoét một hình tròn ở giữa có bán kính r(cm),r < R. a) Viết công thức tính diện tích phần còn lại của miếng bia. b) Tính diện tích phần còn lại của miếng bia biết tổng hai bán kính là 10 cm và hiệu hai bán kính là 3 cm. $ Hình ...

[2] Problem 3.9 (Page: 34)
    Type: geometry
    Figure relevance: 0.58
    Figures: Cho hình
    Question (normalized): . Cho tam giác ABC vuông cân tại đ//nh A. Ghép thêm vào phía ngoài tam giác đó tam giác 8CD vuông cân tại đ//nh B. Chứng minh tứ giác A8ÖC là một hình thang vuông (hình thang có một cạnh bên vuông góc với hai đáy). 3 nhân 1 độ. Cho hình thang cân ABCDvới hai đường thẳng chứa hai cạnh bên AD, BC cắt ...

[3] Problem 5.11 (Page: 34)
    Type: geometry
    Figure relevance: 0.55
    Figures: Cho hình
  

# Save Results with PhoBERT Features

In [25]:
output_folder = "json_output"
os.makedirs(output_folder, exist_ok=True)

# Filter: ONLY geometry problems WITH figures (relevance >= 0.3 AND has figure refs)
figure_threshold = 0.3
geometry_problems = [
    p for p in all_problems 
    if p.get('ai_problem_type') == 'geometry'
    and p.get('figure_relevance', 0) >= figure_threshold
    and len(p.get('figures', [])) > 0
]

print(f"  Total problems extracted: {len(all_problems)}")
print(f"  Geometry + Figures (relevance >= {figure_threshold}): {len(geometry_problems)}")
print(f"  Filtered out: {len(all_problems) - len(geometry_problems)}")

# Prepare data structure
output_data = {
    "metadata": {
        "pdf_path": pdf_file_path,
        "total_pages": len(page_texts),
        "total_problems": len(geometry_problems),
        "problems_with_solution": sum(1 for p in geometry_problems if p.get('has_solution', False)),
        "problems_without_solution": sum(1 for p in geometry_problems if not p.get('has_solution', False)),
        "processed_at": datetime.now().isoformat(),
        "processing_method": "PhoBERT + Tesseract",
        "filter": "geometry_only",
        "model_info": {
            "phobert": "vinai/phobert-base",
            "ocr": "tesseract-vie"
        }
    },
    "problems": []
}

# Save ONLY geometry problems with PhoBERT features
for i, prob in enumerate(geometry_problems, 1):
    output_data["problems"].append({
        "id": i,
        "problem_number": prob['problem_number'],
        "page_number": prob.get('page_number'),
        
        # Normalized versions only (plain text for model training)
        "question": normalize_for_training(prob['question']),
        "solution": normalize_for_training(prob.get('solution', '')),
        
        "figure_references": prob['figures'],
        "has_solution": prob.get('has_solution', False),
        
        # PhoBERT analysis
        "ai_features": {
            "problem_type": prob.get('ai_problem_type'),
            "type_confidence": prob.get('type_confidence', {}),
            "figure_relevance": prob.get('figure_relevance', 0),
            "embedding_preview": prob.get('embedding', [])[:10]  # First 10 dimensions
        }
    })

# Save main results
output_file = os.path.join(output_folder, "problems_with_figures_phobert.json")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"Folder: {output_folder}")
print(f"File: problems_with_figures_phobert.json")
print(f"\nStatistics:")
print(f"  Total problems: {output_data['metadata']['total_problems']}")
print(f"  ├─ With solution: {output_data['metadata']['problems_with_solution']}")
print(f"  └─ Without solution: {output_data['metadata']['problems_without_solution']}")
print(f"  Total pages: {output_data['metadata']['total_pages']}")
print(f"\n  Features: PhoBERT embeddings, problem classification, figure relevance")

  Total problems extracted: 113
  Geometry + Figures (relevance >= 0.3): 3
  Filtered out: 110
Folder: json_output
File: problems_with_figures_phobert.json

Statistics:
  Total problems: 3
  ├─ With solution: 0
  └─ Without solution: 3
  Total pages: 113

  Features: PhoBERT embeddings, problem classification, figure relevance


# Similarity Search Demo

In [ ]:
def find_similar_problems(query_text, problems, top_k=5):
    """
    Find similar problems using PhoBERT embeddings
    """
    query_emb = get_text_embedding(query_text)
    
    similarities = []
    for prob in problems:
        prob_emb = np.array(prob['embedding']).reshape(1, -1)
        similarity = cosine_similarity(query_emb, prob_emb)[0][0]
        similarities.append({
            'problem': prob,
            'similarity': float(similarity)
        })
    
    similarities.sort(key=lambda x: x['similarity'], reverse=True)
    return similarities[:top_k]


# Demo: Find problems similar to a query
if len(all_problems) > 0:
    print("\n🔍 SIMILARITY SEARCH DEMO")
    print("="*60)
    
    query = "Tính diện tích tam giác dựa vào hình vẽ"
    print(f"Query: {query}\n")
    
    similar = find_similar_problems(query, all_problems, top_k=3)
    
    for i, item in enumerate(similar, 1):
        prob = item['problem']
        sim = item['similarity']
        print(f"[{i}] Similarity: {sim:.3f}")
        print(f"    Problem: {prob['problem_number']} (Page {prob['page_number']})")
        print(f"    Type: {prob['ai_problem_type']}")
        print(f"    Question: {prob['question'][:150]}...")
        print()
else:
    print("\n⚠️ No problems found for similarity search demo")

In [ ]:
# Test normalization function
sample_text_with_symbols = """
Cho △ABC có ∠A = 60°, ∠B = 45°
Tính ∠C = 180° − 60° − 45° = 75°
Tam giác ABC có BC × sin A ≥ 10
Đáp án: ≈ 8.66
"""

print("ORIGINAL TEXT (with symbols):")
print(sample_text_with_symbols)
print("\n" + "="*60 + "\n")

print("NORMALIZED TEXT (for training):")
normalized = normalize_for_training(sample_text_with_symbols)
print(normalized)

ORIGINAL TEXT (with symbols):

Cho △ABC có ∠A = 60°, ∠B = 45°
Tính ∠C = 180° − 60° − 45° = 75°
Tam giác ABC có BC × sin A ≥ 10
Đáp án: ≈ 8.66



NORMALIZED TEXT (for training):


NameError: name 'normalize_for_training' is not defined